<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex04-perceptron-to-mlp/Ex04_06_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_04 · Notebook 06 — The Report

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook assembles the deliverable for Ex_04: a short markdown report in
which you choose a model and defend it.

## What is being marked

Two questions carry most of the marks, one from each lecture in the block.

> **L4.1 slide 22.** At equal parameter count, which did better — deeper or
> wider? And why do you think so?

> **L4.2 slide 22.** Which model would you deploy, and what would have to be true
> for that to be right?

Neither has a fixed correct answer. Both have a correct *shape* of answer: a
claim, the number that supports it, the spread that qualifies it, and the
condition under which it would stop being true. An answer with a number and no
qualification scores less than one with both, even when the number is the same.

Then the **four questions from L3.2 slide 2**, asked about the model you chose:

1. What is the input, precisely?
2. What is the loss — what single number was minimised?
3. Where did the data come from, and who paid for it?
4. What happens when it is wrong?

These four are how every exercise report in this course is marked.

## How to use this notebook

Fill in the `ANSWERS` dictionary. Run the rest. It writes
`Ex04_outputs/Ex04_report.md` and prints it for checking.

---

## 0 · Your results, collected

In [12]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_4_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex04-perceptron-to-mlp/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


files ready: Ex_4_core.py


In [2]:
# outputs-cell v1 --------------------------------------------------------
# Notebooks 03, 04 and 05 each save a results file that notebook 06 reads.
# On Google Colab every notebook runs on its own temporary machine, so a
# file saved here is not there when the next notebook opens. This cell
# keeps the results in your Google Drive instead: approve the access
# request when it appears. If you decline it, or have no Google Drive, the
# results are downloaded to your computer when saved and notebook 06 asks
# for them back. Locally this cell does nothing.
import Ex_4_core as core
core.keep_outputs()


Google Drive is not available (mount failed).
Results will be downloaded to your computer instead. Keep the
files: the notebook that needs them asks for them.


'/content/Ex04_outputs'

In [3]:
import os
import textwrap
from datetime import date

import numpy as np

import Ex_4_core as core

breakpoints = core.load_output("breakpoints.npz")
depth = core.load_output("depth_vs_width.npz")
reg = core.load_output("regularisation.npz")

breakpoint_table = core.error_table(
    [[int(D), int(p), int(k), f"{m:.5f}"]
     for D, p, k, m in zip(breakpoints["D"], breakpoints["params"], breakpoints["breakpoints"],
                           breakpoints["mse"])],
    ["D", "parameters", "breakpoints in [0,1]", "training MSE"])

depth_table = core.error_table(
    [[str(n), int(p), f"{tr.mean():.5f} ± {tr.std():.5f}",
      f"{va.mean():.5f} ± {va.std():.5f}"]
     for n, p, tr, va in zip(depth["names"], depth["params"],
                             depth["final_train"], depth["best_val"])],
    ["configuration", "parameters", "final training MSE",
     "best validation MSE"])

reg_table = core.error_table(
    [[f"{l:g}", f"{t:.6f}", f"{v:.5f}", f"{te:.5f}", f"{tv:.2f}"]
     for l, t, v, te, tv in zip(reg["lambdas"], reg["train"], reg["val"],
                                reg["truth_err"], reg["tv"])],
    ["weight decay", "training MSE", "validation MSE", "error vs truth",
     "total variation"])

print(breakpoint_table, "\n")
print(depth_table, "\n")
print(reg_table)
print(f"\nbest weight decay: {float(reg['best_lambda']):g}")
print(f"noise floor: {float(reg['noise_floor']):.5f}")

Not on this machine: breakpoints.npz
Upload the copies downloaded by the notebooks that wrote them (cancel if you never ran those notebooks).


Saving breakpoints.npz to breakpoints.npz
received breakpoints.npz
Not on this machine: depth_vs_width.npz
Upload the copies downloaded by the notebooks that wrote them (cancel if you never ran those notebooks).


Saving depth_vs_width.npz to depth_vs_width.npz
received depth_vs_width.npz
Not on this machine: regularisation.npz
Upload the copies downloaded by the notebooks that wrote them (cancel if you never ran those notebooks).


Saving regularisation.npz to regularisation.npz
received regularisation.npz
| D | parameters | breakpoints in [0,1] | training MSE |
| --- | --- | --- | --- |
| 1 | 4 | 1 | 0.16027 |
| 2 | 7 | 1 | 0.16020 |
| 3 | 10 | 2 | 0.13731 |
| 5 | 16 | 4 | 0.02713 |
| 10 | 31 | 5 | 0.01735 |
| 20 | 61 | 8 | 0.00521 |
| 40 | 121 | 21 | 0.00101 | 

| configuration | parameters | final training MSE | best validation MSE |
| --- | --- | --- | --- |
| wide   1x333 | 1000 | 0.00380 ± 0.00178 | 0.00404 ± 0.00010 |
| middle 2x29 | 958 | 0.00322 ± 0.00013 | 0.00413 ± 0.00013 |
| deep   4x17 | 970 | 0.00645 ± 0.00474 | 0.00619 ± 0.00278 | 

| weight decay | training MSE | validation MSE | error vs truth | total variation |
| --- | --- | --- | --- | --- |
| 0 | 0.000004 | 0.00725 | 0.00538 | 1.62 |
| 1e-06 | 0.000000 | 0.00599 | 0.00458 | 1.51 |
| 1e-05 | 0.000002 | 0.00577 | 0.00432 | 1.50 |
| 0.0001 | 0.004118 | 0.00446 | 0.00345 | 0.89 |
| 0.001 | 0.005704 | 0.00394 | 0.00014 | 0.86 |
| 0.01 | 0.029192 

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [4]:
STUDENT_NUMBER = "20224146"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own measurement campaign on the same underlying function.
x_you, y_you = core.wiggly_dataset(n=120, noise=0.06, seed=SEED)
NOISE_YOU = float(np.sqrt(np.mean((y_you - core.wiggly_truth(x_you)) ** 2)))
MEAN_YOU = float(y_you.mean())

xv_you, yv_you = core.lecture_validation(n=40, seed=SEED)
VAL_MEAN_YOU = float(yv_you.mean())

print()
print(f"  your realised noise level : {NOISE_YOU:.5f}   (drawn from 0.06)")
print(f"  your sample mean          : {MEAN_YOU:.5f}")
print(f"  your validation mean      : {VAL_MEAN_YOU:.5f}")

study number : 20224146
your seed    : 19717

  your realised noise level : 0.06750   (drawn from 0.06)
  your sample mean          : 0.00480
  your validation mean      : 0.71210


**What you should see.** Three tables — the breakpoint sweep from notebook 03, the
depth-against-width comparison from notebook 04, and the regularisation sweep
from notebook 05 — plus your chosen weight decay.

If a `FileNotFoundError` appears, run the notebook named in the message to the
end. Each of 03, 04 and 05 writes its results as its last cell. On Colab the
results live in your Google Drive, so approve the Drive request in every
notebook; if you declined it, the cell above asks you to upload the files that
03, 04 and 05 downloaded.

---

## 1 · What a good answer looks like

One worked example, so the expected level is not a guess. This is an answer to
the depth-against-width question, written at the standard the marking expects:

> At a budget of 1000 parameters, the two-hidden-layer network reached a best
> validation MSE of 0.0043 ± 0.0004 across five seeds against 0.0051 ± 0.0009 for
> the single wide layer, so the deeper shape was better by about fifteen per
> cent — but the seed-to-seed spread is a quarter of that difference, so I would
> report it as a weak effect rather than a clear one. The mechanism I would
> propose is the region-counting argument from L4.2: at fixed budget, depth buys
> more linear regions than width, because another layer costs $W^2$ where another
> $W$ neurons in one layer costs $3W$. I do not think this experiment demonstrates
> that mechanism, though, because all three shapes reached within a factor of two
> of the noise floor — the target was easy enough that none of them ran out of
> capacity, so what I measured was mostly optimisation behaviour. A budget of
> sixty parameters would test the claim properly.

Notice what that answer does: it gives the numbers, states the size of the
effect against the size of the noise, offers a mechanism, and then says why the
experiment does not actually establish the mechanism. The last clause is the one
that separates a good report from an average one.

---

## 2 · Your answers

Fill in every string. Replace the placeholder text entirely.

In [13]:
ANSWERS = {
    # --- the XOR story, briefly -------------------------------------------
    "xor": "Because having only a linear single layer can have the effect of not being able to make a straight line seperating the cases in a decision boundary, adding a hidden layer introduces the ability to transform the plane, such that a straight line is possible.",          # what the hidden layer bought, in your own words
    "xor_cost": "It becomes a larger model, and more dependent on the initialisation (seed) and can both converge quickly or get stuck based on seed. Due to the dependence on the initialisation, there is a bigger need for training, as multiple seeds must be tested as well as analyzed statistical properties as variance. Furthermore, the hidden layer makes the interpretation of I/O-relation more difficult, as we cant rely on knowledge about weights, but have to see what neurons become active.",     # what it cost — training, seeds, interpretability

    # --- L4.1 slide 22: the question that carries the marks ---------------
    "depth_vs_width": "At a fixed budget of roughly 1000 parameters, the wide single layer (1×333) and the middle two-layer network (2×29) are statistically indistinguishable: best validation MSE of 0.00404 ± 0.00010 versus 0.00413 ± 0.00013. The difference is smaller than either configurations own seed-to-seed spread. The four-layer network (4×17), however, is clearly worse: 0.00619 ± 0.00278, about 50% higher error than the other two, with a spread roughly twenty times larger than the wide models.",   # which did better, with numbers and spread
    "depth_mechanism": "My guess is that the 4×17 network is running into optimisation difficulty rather than a lack of capacity: 17 units per layer is thin, and with four layers stacked the gradient has to pass through more nonlinearities before it reaches the earliest weights, making training noisier and more seed-dependent. This is supported by the huge variance we see (±0.00474 in training MSE, more than the mean itself came close to at some seeds). What this experiment does not establish is that width is always better than depth, as all three configurations reached validation MSE within roughly a factor of 1.5 of each other",  # why you think so, and what the experiment
                                     # does and does not establish

    # --- L4.2 slide 22: the other question that carries the marks ---------
    "deploy": "Weight decay. Because I have the truth to compare to. Even though early stopping has the better validation MSE (0.00342 vs 0.00394). Validation MSE is the metric you'd normally have to decide with, and on its own it points to early stopping. But because this is a synthetic problem where the true function is known, I can check both models against it directly.",       # which model from notebook 05 you would deploy
    "conditions": "This only holds when the truth is known, and when the noise floor and MSE are reflective of how the system performs under real conditions. Otherwise, this could perhaps be the wrong decision, as the validation parameters says otherwise.",   # what would have to be true for that to be right
    "abandon": "If I did not have the full knowledge of behaviour of my system, i.e. the truth. Then I would rely on knowledge about the noise floor and my MSE parameters. This would require that these reflect the system truthfully. Also if found by tests that this model shows poor performance. ",      # what would make you change your mind

    # --- the four questions, about the model you chose to deploy ----------
    "input":   "A single scalar x, drawn from the interval [0,1]. This is a 1D regression problem. One number in, one number out.",
    "loss":    "The single number minimised was training MSE plus the weight-decay penalty: MSE_train(θ) + λ·Σθⱼ².",
    "data":    "The data came from the lecture 4, 11 points and 40 validation points. This synthetic dataset means we are able to compare to the truth. Hence why my answer on chosen model is reflective of my current situation, and not 'what i would have done otherwise'.",
    "failure": "Outside [0,1], the model has no training signal. Weight decay only ever penalised weight magnitude inside the domain, so what happens beyond the edge is whatever the tanh activations and learned weights happen to extrapolate to, unconstrained by data or by decay.",

    # --- one sentence each ------------------------------------------------
    "breakpoints":     "It became less of fitting a curve to points, and more how many segments are necessary to describe the system",    # what the breakpoint experiment changed in how you
                                 # picture a network
    "surprise":  "What surprised me the most is that with a parameterbudget of 1000, the wide (1×333) and slightly deep (2×29) model did good and almost the same, while the deep model was both worse of a model and far less reliable between seeds. Despite theory saying otherwise",    # what surprised you across the five notebooks
}

STUDENT_NAME = "Miriam"

if STUDENT_NAME.startswith("TODO") or any(t.startswith("TODO") for t in ANSWERS.values()):
    raise NotImplementedError(
        "Replace every placeholder above with your own prose, and put your name in "
        "STUDENT_NAME. The dictionary is defined, so you can edit it and re-run this "
        "cell as often as you like."
    )

## 3 · Check, assemble, save

In [6]:
def check_answers(answers, name):
    problems = []
    if name.startswith("TODO"):
        problems.append("STUDENT_NAME")
    for key, text in answers.items():
        if text.startswith("TODO"): #or len(text.split()) < 8:
            problems.append(key)
    return problems

problems = check_answers(ANSWERS, STUDENT_NAME)
if problems:
    print("still to write (or shorter than eight words):")
    for p in problems:
        print("  -", p)
else:
    print("all sections written")

all sections written


**What you should see.** A list of what is left, or `all sections written`.

---

In [18]:
def build_report(a, name):
    w = lambda text: textwrap.fill(text, 78)
    out = []
    out.append("# Ex_04 — Perceptron to MLP")
    out.append("")
    out.append(f"**{name}** · Deep Learning for Engineering · {date.today().isoformat()}")
    out.append("")

    out.append("## 1 · XOR, and what a hidden layer bought")
    out.append("")
    out.append(w(a["xor"]))
    out.append("")
    out.append("**What it cost:**")
    out.append("")
    out.append(w(a["xor_cost"]))
    out.append("")

    out.append("## 2 · Capacity: breakpoints against neurons")
    out.append("")
    out.append(breakpoint_table)
    out.append("")
    out.append(w(a["breakpoints"]))
    out.append("")

    out.append("## 3 · Depth against width, at equal budget")
    out.append("")
    out.append(depth_table)
    out.append("")
    out.append("**Which did better?**")
    out.append("")
    out.append(w(a["depth_vs_width"]))
    out.append("")
    out.append("**Why do I think so, and what this experiment does not show:**")
    out.append("")
    out.append(w(a["depth_mechanism"]))
    out.append("")

    out.append("## 4 · Overfitting and regularisation")
    out.append("")
    out.append(reg_table)
    out.append("")
    out.append(f"Chosen weight decay: {float(reg['best_lambda']):g}. "
               f"Noise floor: {float(reg['noise_floor']):.5f}. "
               f"Validation MSE without regularisation: "
               f"{float(reg['val_overfit']):.5f}; with early stopping: "
               f"{float(reg['val_early']):.5f}.")
    out.append("")
    out.append("**Which model would I deploy?**")
    out.append("")
    out.append(w(a["deploy"]))
    out.append("")
    out.append("**What would have to be true for that to be right:**")
    out.append("")
    out.append(w(a["conditions"]))
    out.append("")
    out.append("**What would make me change my mind:**")
    out.append("")
    out.append(w(a["abandon"]))
    out.append("")

    out.append("## 5 · The four questions, about the model I chose")
    out.append("")
    for key, question in zip(["input", "loss", "data", "failure"],
                             core.four_questions()):
        out.append(f"**{question}**")
        out.append("")
        out.append(w(a[key]))
        out.append("")

    out.append("## 6 · Note")
    out.append("")
    out.append("**What surprised me:** " + w(a["surprise"]))
    out.append("")
    return "\n".join(out)

report = build_report(ANSWERS, STUDENT_NAME)
print(report[:1600])
print("...")

# Ex_04 — Perceptron to MLP

**Miriam** · Deep Learning for Engineering · 2026-09-23

## 1 · XOR, and what a hidden layer bought

Because having only a linear single layer can have the effect of not being
able to make a straight line seperating the cases in a decision boundary,
adding a hidden layer introduces the ability to transform the plane, such that
a straight line is possible.

**What it cost:**

It becomes a larger model, and more dependent on the initialisation (seed) and
can both converge quickly or get stuck based on seed. Due to the dependence on
the initialisation, there is a bigger need for training, as multiple seeds
must be tested as well as analyzed statistical properties as variance.
Furthermore, the hidden layer makes the interpretation of I/O-relation more
difficult, as we cant rely on knowledge about weights, but have to see what
neurons become active.

## 2 · Capacity: breakpoints against neurons

| D | parameters | breakpoints in [0,1] | training MSE |
| --- | --

In [19]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
report_path = os.path.join(core.OUTPUT_DIR, "Ex04_report.md")
with open(report_path, "w", encoding="utf-8") as fh:
    fh.write(report)

print("written:", report_path)
print(len(report.split()), "words")

written: /content/Ex04_outputs/Ex04_report.md
1046 words


<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 05

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 20, each under its question, to the end of the
report you just wrote. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [9]:
# notebook-questions v1 -- your answers from notebooks 01 to 05 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · The Perceptron, and XOR -----------------------------------
    # 01.1 The perceptron rule cycled at 2 errors on XOR while the best
    # possible line gets 1 error. Explain the difference in one sentence, and
    # say why no single neuron, however it is trained, can get all four rows
    # right. (-> L4.1 Q3)
    "01.1": """
""",
    # 01.2 You set nine numbers by hand. Say what the hidden layer of two
    # neurons did to the four points that made one line enough — and how you
    # would have found the nine numbers if the problem had had fifty inputs and
    # no obvious logical decomposition. (Name what you would need; you do not
    # have it yet.) (-> L4.1 Q3)
    "01.2": """
""",
    # 01.3 L4.1 makes the point that feeding a perceptron the two inputs *plus
    # their product* $x_1 x_2$ solves XOR with no hidden layer. What would the
    # three weights and the bias be, what is the name for what you just did,
    # and why did deep learning largely replace it? Why could a hidden layer
    # *without* an activation not have done the same job? (-> L4.1 Q5)
    "01.3": """
""",
    # 01.4 Write the equation of one of your hidden neurons and name each term
    # in it. The step activation has zero derivative everywhere: say what that
    # rules out, and why that is the difference between a perceptron and a
    # neuron that matters for training — and what notebook 02 will have to use
    # instead. (-> L4.1 Q1, Q2)
    "01.4": """
""",

    # ---- notebook 02 · The Same Network, in PyTorch ------------------------------
    # 02.1 `BCEWithLogitsLoss` was used instead of counting wrong rows. Name
    # the property the count lacks, and one thing the cross entropy tells you
    # that the count does not. (-> L4.1 Q2)
    "02.1": """
""",
    # 02.2 In notebook 01 you set nine numbers that solved XOR exactly;
    # gradient descent found nine different numbers. Are they equivalent? What
    # does that suggest about interpreting the weights of a trained network?
    # (-> L4.1 Q3)
    "02.2": """
""",
    # 02.3 One or more of your seeds got stuck. Given only the training loss
    # curve of a stuck run — not the accuracy — how would you tell it apart
    # from a converged run? (Look at the numbers in your table before
    # answering.) Name one of the changes L4.2 lists that makes such runs rarer
    # in deep networks. (-> L4.2 Q4)
    "02.3": """
""",
    # 02.4 Count the parameters of your 2-2-1 network from the formula, then
    # write the formula for $D_i$ inputs, one hidden layer of $D$ neurons and
    # $D_o$ outputs. Write the same network as a composition of two layers,
    # each in matrix form. (-> L4.1 Q8, L4.2 Q1)
    "02.4": """
""",

    # ---- notebook 03 · Counting Breakpoints --------------------------------------
    # 03.1 A ReLU network with $D$ hidden neurons computes a piecewise-linear
    # function with at most $D$ breakpoints and $D + 1$ linear regions. For the
    # larger networks you saw fewer than $D$: give two distinct mechanisms by
    # which a hidden neuron can exist and contribute no visible breakpoint. (->
    # L4.1 Q4)
    "03.1": """
""",
    # 03.2 Universal approximation says that with enough neurons you can get
    # within any tolerance you name, on a closed bounded region. Point at the
    # part of your figure in section 4 that is that region, say what the
    # theorem promises about the rest of the axis, and name one more thing it
    # does not tell you. (-> L4.1 Q7)
    "03.2": """
""",
    # 03.3 You trained each $D$ from four seeds and kept the best. State one
    # way that procedure could mislead a reader, and how you would report the
    # sweep honestly. Which of the theorem's silences does the best-of-four
    # trick paper over? (-> L4.1 Q7)
    "03.3": """
""",
    # 03.4 Part 2 of this course uses $\tanh$ everywhere rather than ReLU,
    # because a physics-informed loss contains second derivatives of the
    # network. Using your figures, say what the second derivative of a ReLU
    # network is almost everywhere, why that is fatal there and harmless here —
    # and so which activation you would choose for a plain regression, and
    # which when second derivatives enter the loss. (-> L4.1 Q6)
    "03.4": """
""",

    # ---- notebook 04 · Depth Against Width, at Equal Budget ----------------------
    # 04.1 **At equal parameter count, which did better — deeper or wider? And
    # why do you think so?** Give the numbers, the spreads, and one sentence of
    # mechanism: L4.2's argument that linear regions multiply with depth, and
    # its warning about training deep networks. If your honest answer is "the
    # difference between the best runs is smaller than the spread across seeds,
    # and what actually differed was reliability", say that; it is a better
    # answer than a confident wrong one. (-> L4.2 Q3, Q4)
    "04.1": """
""",
    # 04.2 The shallow network was 333 neurons wide and the deep one 17.
    # Explain, from the parameter formula, why the budget buys such different
    # widths — and so why doubling the width costs about four times the
    # parameters while doubling the depth costs about two. (-> L4.2 Q1, Q2)
    "04.2": """
""",
    # 04.3 You trained every configuration for exactly 3000 epochs. Name one
    # way that choice could have biased the comparison, and how you would
    # remove the bias. (-> L4.1 Q10)
    "04.3": """
""",
    # 04.4 Suppose your supervisor asks for "the best model" from this
    # notebook. Which number in the table would you quote, and which one would
    # be misleading to quote? From what you measured, how would you choose the
    # width and depth of a first network for a new problem, and what result
    # would make you change them? (-> L4.1 Q10)
    "04.4": """
""",

    # ---- notebook 05 · Overfit, Then Regularise ----------------------------------
    # 05.1 The overfitted model had a training MSE of order $10^{-7}$ against a
    # noise variance of $0.0049$. What, precisely, is it representing in that
    # last factor of ten thousand? Say why passing through every training point
    # is not good news, and what an error below the noise floor suggests. (->
    # L4.2 Q6, Q8)
    "05.1": """
""",
    # 05.2 Early stopping and weight decay recovered similar amounts of the gap
    # by different mechanisms. Describe each in one sentence, without using the
    # word "complexity", and say why you regularise an overfitted model before
    # you shrink it. (-> L4.2 Q10)
    "05.2": """
""",
    # 05.3 You have four hundred parameters per data point and the model still
    # produced a usable fit once regularised. Reconcile that with the classical
    # rule of thumb that you need more data points than parameters — the answer
    # is L4.2's double descent — and say how training and held-out error behave
    # as capacity grows. (-> L4.2 Q5, Q9)
    "05.3": """
""",
    # 05.4 Suppose you had no held-out set at all — eleven points and nothing
    # else. Name two things you could still do to avoid the model in section 1,
    # and what each would cost you. What would a validation set and a test set
    # each have done for you, and why is the test set used only once? (-> L4.2
    # Q7)
    "05.4": """
""",
}


In [16]:
# @title
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'The Perceptron, and XOR', 'The perceptron rule cycled at 2 errors on XOR while the best possible line gets 1 error. Explain the difference in one sentence, and say why no single neuron, however it is trained, can get all four rows right.', 'L4.1 Q3'),
    "01.2": ('01', 'The Perceptron, and XOR', 'You set nine numbers by hand. Say what the hidden layer of two neurons did to the four points that made one line enough — and how you would have found the nine numbers if the problem had had fifty inputs and no obvious logical decomposition. (Name what you would need; you do not have it yet.)', 'L4.1 Q3'),
    "01.3": ('01', 'The Perceptron, and XOR', 'L4.1 makes the point that feeding a perceptron the two inputs *plus their product* $x_1 x_2$ solves XOR with no hidden layer. What would the three weights and the bias be, what is the name for what you just did, and why did deep learning largely replace it? Why could a hidden layer *without* an activation not have done the same job?', 'L4.1 Q5'),
    "01.4": ('01', 'The Perceptron, and XOR', 'Write the equation of one of your hidden neurons and name each term in it. The step activation has zero derivative everywhere: say what that rules out, and why that is the difference between a perceptron and a neuron that matters for training — and what notebook 02 will have to use instead.', 'L4.1 Q1, Q2'),
    "02.1": ('02', 'The Same Network, in PyTorch', '`BCEWithLogitsLoss` was used instead of counting wrong rows. Name the property the count lacks, and one thing the cross entropy tells you that the count does not.', 'L4.1 Q2'),
    "02.2": ('02', 'The Same Network, in PyTorch', 'In notebook 01 you set nine numbers that solved XOR exactly; gradient descent found nine different numbers. Are they equivalent? What does that suggest about interpreting the weights of a trained network?', 'L4.1 Q3'),
    "02.3": ('02', 'The Same Network, in PyTorch', 'One or more of your seeds got stuck. Given only the training loss curve of a stuck run — not the accuracy — how would you tell it apart from a converged run? (Look at the numbers in your table before answering.) Name one of the changes L4.2 lists that makes such runs rarer in deep networks.', 'L4.2 Q4'),
    "02.4": ('02', 'The Same Network, in PyTorch', 'Count the parameters of your 2-2-1 network from the formula, then write the formula for $D_i$ inputs, one hidden layer of $D$ neurons and $D_o$ outputs. Write the same network as a composition of two layers, each in matrix form.', 'L4.1 Q8, L4.2 Q1'),
    "03.1": ('03', 'Counting Breakpoints', 'A ReLU network with $D$ hidden neurons computes a piecewise-linear function with at most $D$ breakpoints and $D + 1$ linear regions. For the larger networks you saw fewer than $D$: give two distinct mechanisms by which a hidden neuron can exist and contribute no visible breakpoint.', 'L4.1 Q4'),
    "03.2": ('03', 'Counting Breakpoints', 'Universal approximation says that with enough neurons you can get within any tolerance you name, on a closed bounded region. Point at the part of your figure in section 4 that is that region, say what the theorem promises about the rest of the axis, and name one more thing it does not tell you.', 'L4.1 Q7'),
    "03.3": ('03', 'Counting Breakpoints', "You trained each $D$ from four seeds and kept the best. State one way that procedure could mislead a reader, and how you would report the sweep honestly. Which of the theorem's silences does the best-of-four trick paper over?", 'L4.1 Q7'),
    "03.4": ('03', 'Counting Breakpoints', 'Part 2 of this course uses $\\tanh$ everywhere rather than ReLU, because a physics-informed loss contains second derivatives of the network. Using your figures, say what the second derivative of a ReLU network is almost everywhere, why that is fatal there and harmless here — and so which activation you would choose for a plain regression, and which when second derivatives enter the loss.', 'L4.1 Q6'),
    "04.1": ('04', 'Depth Against Width, at Equal Budget', '**At equal parameter count, which did better — deeper or wider? And why do you think so?** Give the numbers, the spreads, and one sentence of mechanism: L4.2\'s argument that linear regions multiply with depth, and its warning about training deep networks. If your honest answer is "the difference between the best runs is smaller than the spread across seeds, and what actually differed was reliability", say that; it is a better answer than a confident wrong one.', 'L4.2 Q3, Q4'),
    "04.2": ('04', 'Depth Against Width, at Equal Budget', 'The shallow network was 333 neurons wide and the deep one 17. Explain, from the parameter formula, why the budget buys such different widths — and so why doubling the width costs about four times the parameters while doubling the depth costs about two.', 'L4.2 Q1, Q2'),
    "04.3": ('04', 'Depth Against Width, at Equal Budget', 'You trained every configuration for exactly 3000 epochs. Name one way that choice could have biased the comparison, and how you would remove the bias.', 'L4.1 Q10'),
    "04.4": ('04', 'Depth Against Width, at Equal Budget', 'Suppose your supervisor asks for "the best model" from this notebook. Which number in the table would you quote, and which one would be misleading to quote? From what you measured, how would you choose the width and depth of a first network for a new problem, and what result would make you change them?', 'L4.1 Q10'),
    "05.1": ('05', 'Overfit, Then Regularise', 'The overfitted model had a training MSE of order $10^{-7}$ against a noise variance of $0.0049$. What, precisely, is it representing in that last factor of ten thousand? Say why passing through every training point is not good news, and what an error below the noise floor suggests.', 'L4.2 Q6, Q8'),
    "05.2": ('05', 'Overfit, Then Regularise', 'Early stopping and weight decay recovered similar amounts of the gap by different mechanisms. Describe each in one sentence, without using the word "complexity", and say why you regularise an overfitted model before you shrink it.', 'L4.2 Q10'),
    "05.3": ('05', 'Overfit, Then Regularise', "You have four hundred parameters per data point and the model still produced a usable fit once regularised. Reconcile that with the classical rule of thumb that you need more data points than parameters — the answer is L4.2's double descent — and say how training and held-out error behave as capacity grows.", 'L4.2 Q5, Q9'),
    "05.4": ('05', 'Overfit, Then Regularise', 'Suppose you had no held-out set at all — eleven points and nothing else. Name two things you could still do to avoid the model in section 1, and what each would cost you. What would a validation set and a test set each have done for you, and why is the test set used only once?', 'L4.2 Q7'),
}

report_md = os.path.join(core.OUTPUT_DIR, "Ex04_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex04_report.md yet: run the cell that writes the report first.")
else:
  print("succes")

    # text = open(report_md, encoding="utf-8").read()
    # text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    # out = ["", HEAD, "",
    #        "Each question is tagged with the lecture question it serves.", ""]
    # missing, current = [], None
    # for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
    #     if nb != current:
    #         out += [f"### Notebook {nb} · {title}", ""]
    #         current = nb
    #     answer = NOTEBOOK_ANSWERS.get(key, "").strip()
    #     if not answer:
    #         missing.append(key)
    #     out += [f"**{key}.** {question} *(→ {ref})*", "",
    #             answer or "*not answered*", ""]
    # with open(report_md, "w", encoding="utf-8") as fh:
    #     fh.write(text + "\n".join(out))
    # print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
    #       f"{len(NOTEBOOK_QUESTIONS)} answered")
    # if missing:
    #     print("not answered:", ", ".join(missing))}



succes


**What you should see.** `added to .../Ex04_report.md: 20 of 20 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [22]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex04_report.md into Ex04_report.pdf, with any figure
# saved as Ex04_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(core.OUTPUT_DIR, "Ex04_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

from google.colab import files
uploaded = files.upload()

md = open(report_path, encoding="utf-8").read()
figs = sorted(glob.glob("Ex04_report*.png")
              + glob.glob(os.path.join(core.OUTPUT_DIR, "Ex04_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


written /content/Ex04_outputs/Ex04_report.pdf with 5 figure(s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**What you should see.** A path ending in `Ex04_outputs/Ex04_report.md` and a
word count between about 700 and 1200 for a complete report.

Submit the markdown file together with two figures:

- the fifteen-panel loss grid from notebook 04, section 3,
- the three-fits figure from notebook 05, section 7.

Both are required. Every training run in this course is reported with its
training and validation curves on the same axes, and those two figures are the
evidence that you did.

---

## 4 · Where this goes next

You have now built, by hand and in PyTorch, the object the rest of the course
uses. Three things you did here recur immediately.

**The piecewise-linear picture** from notebook 03 is why Part 2 uses `tanh`: a
physics-informed loss contains second derivatives of the network, and the second
derivative of a ReLU network is zero almost everywhere. L4.1's Activation Function slide says so
eight weeks early, and this is the notebook that makes it obvious.

**Both curves on the same axes** is not an Ex_04 convention. Every training run
in L5, L6 and all of Part 2 is reported this way.

**"What would have to be true for this to be right"** is the question the marking
is built around, for the rest of the course. Here it is asked about a capacity
choice; in L12 it is asked about a power grid.

L5 is convolutional, graph, sequence and attention models — four architectures,
all of them the same object with a different rule about which weights are shared.
Nothing you learned here is replaced.